# Building Analysis


Here, I am exploring to check patterns, find interesting cases, and audit data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Exploring

**Source-Data Note:**  
The VPD Open Data download with "All Years / All Neighbourhoods" generated a dataset without records from 2022. A separate "2022 ? All Neighbourhoods" download contained the 2022 dta, so the source files will be combined before analysis.

In [2]:
# load

all_years = pd.read_csv("../data/raw/crimedata_csv_AllNeighbourhoods_AllYears.csv")
data_2022 = pd.read_csv("../data/raw/crimedata_csv_AllNeighbourhoods_2022.csv")

all_years.info(), data_2022.info()

all_years.columns.tolist(), data_2022.columns.tolist()

<class 'pandas.DataFrame'>
RangeIndex: 917009 entries, 0 to 917008
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TYPE           917009 non-null  str    
 1   YEAR           917009 non-null  int64  
 2   MONTH          917009 non-null  int64  
 3   DAY            917009 non-null  int64  
 4   HOUR           917009 non-null  int64  
 5   MINUTE         917009 non-null  int64  
 6   HUNDRED_BLOCK  916997 non-null  str    
 7   NEIGHBOURHOOD  916908 non-null  str    
 8   X              916979 non-null  float64
 9   Y              916979 non-null  float64
dtypes: float64(2), int64(5), str(3)
memory usage: 70.0 MB
<class 'pandas.DataFrame'>
RangeIndex: 34323 entries, 0 to 34322
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   TYPE           34323 non-null  str    
 1   YEAR           34323 non-null  int64  
 2   MONTH          34323 n

(['TYPE',
  'YEAR',
  'MONTH',
  'DAY',
  'HOUR',
  'MINUTE',
  'HUNDRED_BLOCK',
  'NEIGHBOURHOOD',
  'X',
  'Y'],
 ['TYPE',
  'YEAR',
  'MONTH',
  'DAY',
  'HOUR',
  'MINUTE',
  'HUNDRED_BLOCK',
  'NEIGHBOURHOOD',
  'X',
  'Y'])

In [3]:
# last check that all_years don't have 2022
all_years["YEAR"].value_counts().sort_index(), data_2022["YEAR"].value_counts()

(YEAR
 2003    57253
 2004    56473
 2005    51685
 2006    48490
 2007    43308
 2008    40640
 2009    36830
 2010    34499
 2011    33342
 2012    34356
 2013    34599
 2014    39183
 2015    40219
 2016    44094
 2017    43196
 2018    44243
 2019    48152
 2020    37519
 2021    32191
 2023    36719
 2024    33792
 2025    32495
 2026    13731
 Name: count, dtype: int64,
 YEAR
 2022    34323
 Name: count, dtype: int64)

In [4]:
df = pd.concat([all_years, data_2022], ignore_index=True)

df["YEAR"].value_counts().sort_index()

YEAR
2003    57253
2004    56473
2005    51685
2006    48490
2007    43308
2008    40640
2009    36830
2010    34499
2011    33342
2012    34356
2013    34599
2014    39183
2015    40219
2016    44094
2017    43196
2018    44243
2019    48152
2020    37519
2021    32191
2022    34323
2023    36719
2024    33792
2025    32495
2026    13731
Name: count, dtype: int64

In [5]:
# check all data was preserved
len(df) == len(all_years) + len(data_2022)

True

In [6]:
df.shape

(951332, 10)

In [7]:
df.columns.tolist()
df.columns = df.columns.str.lower()

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 951332 entries, 0 to 951331
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   type           951332 non-null  str    
 1   year           951332 non-null  int64  
 2   month          951332 non-null  int64  
 3   day            951332 non-null  int64  
 4   hour           951332 non-null  int64  
 5   minute         951332 non-null  int64  
 6   hundred_block  951320 non-null  str    
 7   neighbourhood  951225 non-null  str    
 8   x              951301 non-null  float64
 9   y              951301 non-null  float64
dtypes: float64(2), int64(5), str(3)
memory usage: 72.6 MB


,type,year,month,day,hour,minute,hundred_block,neighbourhood,x,y
0,Theft from Vehicle,2026,5,29,20,0,24XX BIRCH ST,Fairview,490288.3251,5.456857e+06
1,Theft from Vehicle,2024,5,23,17,0,24XX BONNYVALE AVE,Victoria-Fraserview,495831.7011,5.451588e+06
2,Theft from Vehicle,2025,4,8,0,23,24XX BURRARD ST,Kitsilano,489392.3161,5.456880e+06
3,Theft from Vehicle,2024,2,9,17,0,24XX CAMBIE ST,Fairview,491636.2151,5.456792e+06
4,Theft from Vehicle,2024,9,15,11,0,24XX CAMBIE ST,Fairview,491636.2151,5.456792e+06


Note that this dataset has 951,332 observations and 10 variables.

Each data seems to show what type of crime occurred, when it occurred (from year down to minutes) and where (block/neighbourhood and x, y)

There are no missing values in the first 6 variables but it is important to whether they are reliable or not.

In [8]:
df.nunique()

type                 11
year                 24
month                12
day                  31
hour                 24
minute               60
hundred_block     22606
neighbourhood        24
x                156913
y                156871
dtype: int64

Questions:
- How many times does each crime occur?
- How many neighbourhoods?

In [9]:
# Types of crime and number of each that occur
df["type"].value_counts()

type
Theft from Vehicle                                        258096
Other Theft                                               255435
Mischief                                                  121191
Offence Against a Person                                   85765
Break and Enter Residential/Other                          74841
Break and Enter Commercial                                 51187
Theft of Vehicle                                           46318
Theft of Bicycle                                           39322
Vehicle Collision or Pedestrian Struck (with Injury)       18386
Vehicle Collision or Pedestrian Struck (with Fatality)       444
Homicide                                                     347
Name: count, dtype: int64

In [10]:
df["neighbourhood"].value_counts(dropna=False)

neighbourhood
Central Business District    263883
West End                      85025
Strathcona                    62250
Mount Pleasant                59763
Fairview                      56990
Renfrew-Collingwood           52382
Grandview-Woodland            52360
Kitsilano                     45158
Kensington-Cedar Cottage      44745
Hastings-Sunrise              31889
Sunset                        31809
Marpole                       23633
Riley Park                    22087
Victoria-Fraserview           18512
Killarney                     17451
Oakridge                      13702
Kerrisdale                    12158
Dunbar-Southlands             11988
West Point Grey               10078
Arbutus Ridge                  9852
South Cambie                   9611
Shaughnessy                    8936
Stanley Park                   5906
Musqueam                       1057
NaN                             107
Name: count, dtype: int64

Central Business District has much more crimes than others with 263,883 observations. Yet, that does not mean it is very dangerous, rather it might mean that it is more populated which naturally lead to more incidents.

- *Idea:* Case Study Central Business District

In [11]:
missing_summary = pd.DataFrame({
  "missing_count": df.isna().sum(), 
  "missing_percent": df.isna().mean() * 100
})
missing_summary

,missing_count,missing_percent
type,0,0.000000
year,0,0.000000
month,0,0.000000
day,0,0.000000
hour,0,0.000000
minute,0,0.000000
hundred_block,12,0.001261
neighbourhood,107,0.011247
x,31,0.003259
y,31,0.003259


The missingness is negligible with the worst column `neighbourhood` missing at only 0.011%.

## Auditing Year/Month/Day

In [12]:
df[["year", "month", "day", "hour", "minute"]].describe()

,year,month,day,hour,minute
count,951332.000000,951332.000000,951332.000000,951332.000000,951332.000000
mean,2013.444246,6.493936,15.367989,12.265423,15.960760
std,6.878057,3.408116,8.756136,7.470976,18.396658
min,2003.000000,1.000000,1.000000,0.000000,0.000000
25%,2007.000000,4.000000,8.000000,7.000000,0.000000
50%,2014.000000,7.000000,15.000000,14.000000,5.000000
75%,2019.000000,9.000000,23.000000,18.000000,30.000000
max,2026.000000,12.000000,31.000000,23.000000,59.000000


Not obvious errors are shown, such as month 13 or minute 60. Yet, there can still be wrong dates like 29th of February in non-leap year.

In [13]:
# whether the date/time are valid
timestamps = pd.to_datetime(
  df[["year", "month", "day", "hour", "minute"]],
  errors="coerce"
)

timestamps.isna().sum()

np.int64(0)

The `datetime()` checked whether it could turn those variables into a date, time and if there were any errors, it would be turned into NaT.  The result of 0 showed no errors of such.

In [14]:
# check how many crimes occur in each year
df["year"].value_counts().sort_index()

year
2003    57253
2004    56473
2005    51685
2006    48490
2007    43308
2008    40640
2009    36830
2010    34499
2011    33342
2012    34356
2013    34599
2014    39183
2015    40219
2016    44094
2017    43196
2018    44243
2019    48152
2020    37519
2021    32191
2022    34323
2023    36719
2024    33792
2025    32495
2026    13731
Name: count, dtype: int64

In [15]:
# start date and end date of data
timestamps.min(), timestamps.max()

(Timestamp('2003-01-01 00:00:00'), Timestamp('2026-06-05 04:46:00'))

In [16]:
# checking
df[df["year"] == 2026]["month"].value_counts().sort_index()

month
1    2862
2    2813
3    2541
4    2613
5    2671
6     231
Name: count, dtype: int64

Low values in 2026  
- Data starts from 2003-1-1 00:00 to 2026-6-5 04:46. The reason 2026 data is far fewer than others.
- *Important:* If doing analysis by year, need to remember 2026 is an incomplete year

In [17]:
sorted(df["year"].unique())

[np.int64(2003),
 np.int64(2004),
 np.int64(2005),
 np.int64(2006),
 np.int64(2007),
 np.int64(2008),
 np.int64(2009),
 np.int64(2010),
 np.int64(2011),
 np.int64(2012),
 np.int64(2013),
 np.int64(2014),
 np.int64(2015),
 np.int64(2016),
 np.int64(2017),
 np.int64(2018),
 np.int64(2019),
 np.int64(2020),
 np.int64(2021),
 np.int64(2022),
 np.int64(2023),
 np.int64(2024),
 np.int64(2025),
 np.int64(2026)]

## Missing Value Patterns

- They have same amount of NA values. Are x and y always missing together?

In [20]:
((df["x"].isna()) == (df["y"].isna())).all()

np.True_

In [23]:
# rows containing at least one NA
missing_rows = df[df.isna().any(axis=1)]
missing_rows
missing_rows.isna().value_counts()

type   year   month  day    hour   minute  hundred_block  neighbourhood  x      y      date 
False  False  False  False  False  False   False          True           False  False  False    76
                                                                         True   True   False    31
                                           True           False          False  False  False    12
Name: count, dtype: int64

In [31]:
df[df["neighbourhood"].isna()][["type", "year", "hundred_block", "neighbourhood", "x", "y"]].head()

,type,year,hundred_block,neighbourhood,x,y
19719,Vehicle Collision or Pedestrian Struck (with I...,2025,15XX GRANVILLE BRDG,NaN,490358.4244,5457799.194
19724,Vehicle Collision or Pedestrian Struck (with I...,2024,16TH AVE AND CROWN ST,NaN,NaN,NaN
19937,Vehicle Collision or Pedestrian Struck (with F...,2024,ARBUTUS ST / W 21ST AVE,NaN,NaN,NaN
20021,Vehicle Collision or Pedestrian Struck (with I...,2025,99XXXX UNKNOWN,NaN,0.0000,0.000
20022,Vehicle Collision or Pedestrian Struck (with I...,2025,99XXXX UNKNOWN,NaN,0.0000,0.000


In [32]:
df[df["x"].isna()][["type", "year", "hundred_block", "neighbourhood", "x", "y"]].head()

,type,year,hundred_block,neighbourhood,x,y
19724,Vehicle Collision or Pedestrian Struck (with I...,2024,16TH AVE AND CROWN ST,NaN,NaN,NaN
19937,Vehicle Collision or Pedestrian Struck (with F...,2024,ARBUTUS ST / W 21ST AVE,NaN,NaN,NaN
20539,Vehicle Collision or Pedestrian Struck (with I...,2024,E 2ND AVE / MAIN ST,NaN,NaN,NaN
20868,Vehicle Collision or Pedestrian Struck (with I...,2024,E/L 1000 GRANVILLE ST,NaN,NaN,NaN
21401,Vehicle Collision or Pedestrian Struck (with I...,2024,KNIGHT ST AT E 61ST AVE,NaN,NaN,NaN


**Observations of Missing Values**  
- x and y are missing and present together
- 76 Rows: Neighbourhood missing, Coordinates present  
- 31 Rows: Neighbourhood missing, Coordiinates missing
- 12 Rows: Hundred_block missing, Neighbourhood & Coordinates present

## Duplicate Observations Audit

There may be rows that are exact duplicates in all 10 columns.

In [ ]:
# count duplicate data
df.duplicated().sum()

np.int64(36105)

In [45]:
# look at duplicate data
duplicate_rows = df[df.duplicated(keep=False)].sort_values(["year", "month", "day", "hour", "minute", "type"])
duplicate_rows.head()

,type,year,month,day,hour,minute,hundred_block,neighbourhood,x,y,date
837753,Offence Against a Person,2003,1,1,0,0,OFFSET TO PROTECT PRIVACY,West End,0.0,0.0,2003-01-01
840323,Offence Against a Person,2003,1,1,0,0,OFFSET TO PROTECT PRIVACY,Central Business District,0.0,0.0,2003-01-01
842182,Offence Against a Person,2003,1,1,0,0,OFFSET TO PROTECT PRIVACY,Central Business District,0.0,0.0,2003-01-01
842294,Offence Against a Person,2003,1,1,0,0,OFFSET TO PROTECT PRIVACY,Central Business District,0.0,0.0,2003-01-01
844267,Offence Against a Person,2003,1,1,0,0,OFFSET TO PROTECT PRIVACY,West End,0.0,0.0,2003-01-01


In [43]:
duplicate_rows.shape

(55301, 11)

In [44]:
duplicate_rows["type"].value_counts()

type
Offence Against a Person    55289
Other Theft                    10
Mischief                        2
Name: count, dtype: int64

Number of Duplicate Observations 
- `df.duplicated()` treats the first row as the originl, marking it `False`
- `df.duplicated(keep=False)` keeps the first row as well ("keep the one marked False")
- There are **19,196 groups of duplicate records**: (rows of `duplicate_rows`) - `df.duplicated().sum()` = 19,196



### Offences Against a Person
- VPD tells that the category aggregates several kinds of violent incidents to reduce Personal Identificable Information (PII) (*Do cite @openvpd)

In [46]:
person_offences = df[df["type"] == "Offence Against a Person"]
person_offences[["hour", "minute", "hundred_block", "x", "y"]].nunique()

hour             1
minute           1
hundred_block    1
x                1
y                1
dtype: int64

In [47]:
person_offences["hundred_block"].value_counts()

hundred_block
OFFSET TO PROTECT PRIVACY    85765
Name: count, dtype: int64

In [48]:
person_offences[["hour", "minute", "x", "y"]].drop_duplicates()

,hour,minute,x,y
50742,0,0,0.0,0.0


Due to privacy protection, all the data one these variables are suppressed.
- *Important:* Never use `hour`, `minute`, `hundred_block`, `x`, or `y` from `Offence Aginst a Person` as if they were real observations

### Other Duplicates (Not Offence Against a Person)

In [53]:
other_duplicates = duplicate_rows[duplicate_rows["type"] != "Offence Against a Person"]
other_duplicates.info()
other_duplicates

<class 'pandas.DataFrame'>
Index: 12 entries, 908155 to 103936
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   type           12 non-null     str           
 1   year           12 non-null     int64         
 2   month          12 non-null     int64         
 3   day            12 non-null     int64         
 4   hour           12 non-null     int64         
 5   minute         12 non-null     int64         
 6   hundred_block  12 non-null     str           
 7   neighbourhood  12 non-null     str           
 8   x              12 non-null     float64       
 9   y              12 non-null     float64       
 10  date           12 non-null     datetime64[us]
dtypes: datetime64[us](1), float64(2), int64(5), str(3)
memory usage: 1.1 KB


,type,year,month,day,hour,minute,hundred_block,neighbourhood,x,y,date
908155,Other Theft,2017,6,13,23,30,10XX DAVIE ST,West End,490565.3884,5.458520e+06,2017-06-13
908158,Other Theft,2017,6,13,23,30,10XX DAVIE ST,West End,490565.3884,5.458520e+06,2017-06-13
908156,Other Theft,2017,6,14,0,0,10XX DAVIE ST,West End,490565.3884,5.458520e+06,2017-06-14
908159,Other Theft,2017,6,14,0,0,10XX DAVIE ST,West End,490565.3884,5.458520e+06,2017-06-14
925872,Other Theft,2022,7,22,12,20,7XX DUNSMUIR ST,Central Business District,491512.5072,5.459023e+06,2022-07-22
925908,Other Theft,2022,7,22,12,20,7XX DUNSMUIR ST,Central Business District,491512.5072,5.459023e+06,2022-07-22
943929,Mischief,2022,9,18,22,0,29XX E HASTINGS ST,Hastings-Sunrise,496808.7101,5.458717e+06,2022-09-18
943933,Mischief,2022,9,18,22,0,29XX E HASTINGS ST,Hastings-Sunrise,496808.7101,5.458717e+06,2022-09-18
100697,Other Theft,2024,8,23,20,35,7XX DUNSMUIR ST,Central Business District,491512.5072,5.459023e+06,2024-08-23
100774,Other Theft,2024,8,23,20,35,7XX DUNSMUIR ST,Central Business District,491512.5072,5.459023e+06,2024-08-23


In [52]:
other_duplicates["date"].value_counts()

date
2017-06-13    2
2017-06-14    2
2022-07-22    2
2022-09-18    2
2024-08-23    2
2025-01-19    2
Name: count, dtype: int64

However, since each has no unique incident ID, it is hard to know if two distinct incidents happened to share time and area, or the same record was accidentally duplicated.

**Observations on Duplicates:**   
- 36,105 apparent duplicates; 19,196 duplicated records
- Mostly due to privacy protection (19,190)
- Other 6 are pairs (total 12) of other crimes but cannot be proven as an error
- Thus, did not drop

## Geographic Audit

VPD map service says the x, y coordinate system is UTM Zone 10N, NAD83 (in metres) (*Do citations https://opendata.vancouver.ca/pages/portal-news/)

### (0, 0) Coordinate Values

In [54]:
# privacy-keeping corrdinates
((df["x"] == 0) & (df["y"] == 0)).sum()

np.int64(86117)

In [55]:
df[(df["x"] == 0) & (df["y"] == 0)]["type"].value_counts()

type
Offence Against a Person                                85765
Homicide                                                  347
Vehicle Collision or Pedestrian Struck (with Injury)        5
Name: count, dtype: int64

### Coordinates for Use

In [57]:
geo_valid = df[df["x"].notna() & df["y"].notna() & ~((df["x"] == 0) & (df["y"] == 0))]
geo_valid[["x", "y"]].describe()

,x,y
count,865184.000000,8.651840e+05
mean,492221.834545,5.456865e+06
std,2610.457546,2.442278e+03
min,388373.000000,5.439487e+06
25%,490739.482000,5.455724e+06
50%,491773.944800,5.457603e+06
75%,493652.598600,5.458772e+06
max,511975.647700,5.512579e+06


**Observations on Coordinates:**   
- For neighbourhood analysis, `Offence Against a Person` data can be included because the `neighbourhood` values are available
- For time(hour & minute)-based analysis, `Offence Against a Person` must be excluded because they have a placeholder 00:00
- For spatial analysis, `Offence Against a Person` must be excluded because the coordinates have a placeholder (0,0)

## Build Dataset

Build variables

In [62]:
# date
df["date"] = pd.to_datetime(df[["year", "month", "day"]], errors="coerce")

# year and month only
df["year_month"] = df["date"].dt.to_period("M").astype(str)

# flag invalid coordinates, neighbourhood, and time
df["valid_coordinates"] = (df["x"].notna() & df["y"].notna() & ~((df["x"] == 0) & (df["y"] == 0)))
df["valid_neighbourhood"] = df["neighbourhood"].notna()
df["valid_time"] = df["type"] != "Offence Against a Person"

# flag incomplete month (note: 2026 is also incomplete but easy to filter)
df["complete_month"] = ~((df["year"] == 2026) & (df["month"] == 6))

Verify the new variables

In [69]:
df[["valid_coordinates", "valid_neighbourhood", "valid_time", "complete_month"]].value_counts()

valid_coordinates  valid_neighbourhood  valid_time  complete_month
True               True                 True        True              864965
False              True                 False       True               85684
                                        True        True                 345
True               True                 True        False                204
False              False                False       True                  54
                                        True        True                  38
                   True                 False       False                 27
True               False                True        True                  15
Name: count, dtype: int64

In [65]:
df["valid_coordinates"].value_counts()

valid_coordinates
True     865184
False     86148
Name: count, dtype: int64

In [66]:
df["valid_neighbourhood"].value_counts()

valid_neighbourhood
True     951225
False       107
Name: count, dtype: int64

In [67]:
df["valid_time"].value_counts()

valid_time
True     865567
False     85765
Name: count, dtype: int64

In [68]:
df["complete_month"].value_counts()

complete_month
True     951101
False       231
Name: count, dtype: int64

### Final check

In [70]:
df.shape

(951332, 16)

In [71]:
df.head()

,type,year,month,day,hour,minute,hundred_block,neighbourhood,x,y,date,year_month,valid_coordinates,valid_neighbourhood,complete_month,valid_time
0,Theft from Vehicle,2026,5,29,20,0,24XX BIRCH ST,Fairview,490288.3251,5.456857e+06,2026-05-29,2026-05,True,True,True,True
1,Theft from Vehicle,2024,5,23,17,0,24XX BONNYVALE AVE,Victoria-Fraserview,495831.7011,5.451588e+06,2024-05-23,2024-05,True,True,True,True
2,Theft from Vehicle,2025,4,8,0,23,24XX BURRARD ST,Kitsilano,489392.3161,5.456880e+06,2025-04-08,2025-04,True,True,True,True
3,Theft from Vehicle,2024,2,9,17,0,24XX CAMBIE ST,Fairview,491636.2151,5.456792e+06,2024-02-09,2024-02,True,True,True,True
4,Theft from Vehicle,2024,9,15,11,0,24XX CAMBIE ST,Fairview,491636.2151,5.456792e+06,2024-09-15,2024-09,True,True,True,True


In [72]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 951332 entries, 0 to 951331
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   type                 951332 non-null  str           
 1   year                 951332 non-null  int64         
 2   month                951332 non-null  int64         
 3   day                  951332 non-null  int64         
 4   hour                 951332 non-null  int64         
 5   minute               951332 non-null  int64         
 6   hundred_block        951320 non-null  str           
 7   neighbourhood        951225 non-null  str           
 8   x                    951301 non-null  float64       
 9   y                    951301 non-null  float64       
 10  date                 951332 non-null  datetime64[us]
 11  year_month           951332 non-null  str           
 12  valid_coordinates    951332 non-null  bool          
 13  valid_neighbourhood  9513

In [73]:
df["year"].value_counts().sort_index()

year
2003    57253
2004    56473
2005    51685
2006    48490
2007    43308
2008    40640
2009    36830
2010    34499
2011    33342
2012    34356
2013    34599
2014    39183
2015    40219
2016    44094
2017    43196
2018    44243
2019    48152
2020    37519
2021    32191
2022    34323
2023    36719
2024    33792
2025    32495
2026    13731
Name: count, dtype: int64

In [75]:
assert df["date"].isna().sum() == 0

# Summary 

In [76]:
df.to_csv("../data/processed/vancouver_crime_analysis.csv", index=False)

What was done?
- Profiled the raw dataset
- Discovered the missing 2022 export problem and combined data programmatically
- Audited missing values and found a pattern; did not drop
- Audited date and time validity: dates are verified and time have 00:00 for privacy
- Audited duplicates: identified privacy-protected fields